# NPPES National Dataset Creator (V2)

**What changed from v1:** The per-state filtering function now collects matching chunks into a Python list and calls `pd.concat` exactly once at the end, instead of calling `pd.concat([filtered_df, filtered_chunk])` inside the chunk loop.

Before saving the initial data files in the States folder, we then clean the data by...
- Mapping Specialties to Taxonomy
- Cleaning City column
- ZipCode to 5 digits
- Adding County

**Why it matters:** The v1 pattern is O(n²) — every concat copies the entire growing DataFrame in memory, so the work-per-chunk grows linearly with how many chunks have already been processed. Collecting frames in a list and concatenating once is O(n) and noticeably faster on the 11GB national file. Output is cleaner with extra data cleaning.

In [1]:
# Importing packages
import pandas as pd
import numpy as np

# Setting relevant file paths
dataDict_path = "/Users/lukebincarousky/Downloads/NPPES/NPPES/Dictionaries/main_nppes_data_dict.csv"
fullNPPES_path = "/Users/lukebincarousky/Downloads/NPPES/NPPES_Data_Dissemination_May_2026_V2/npidata_pfile_20050523-20260510.csv"

# Importing geographic workbooks
usps_geo = pd.read_excel("/Users/lukebincarousky/Downloads/NPPES/NPPES/Geographic Data/ZIP_Locale_Detail.xls")
msa_geo = pd.read_excel("/Users/lukebincarousky/Downloads/NPPES/NPPES/Geographic Data/US State County City Zip MSA workbook.xlsx", sheet_name="USA_MSA")

# Importing Specialty Dictionary
spec_dict = pd.read_excel("/Users/lukebincarousky/Downloads/NPPES/NPPES/Dictionaries/nucc_taxonomy_250.xlsx")

# Saving the list of states as a separate variable
state_list = usps_geo['PHYSICAL STATE'].unique()

# Selecting Relevant Columns

In [2]:
# Importing our data dictionary
data_dict = pd.read_csv(dataDict_path)  # type: ignore

# Creating a list of the columns we need to import using the data dictionary
cols = data_dict['Column_Name'].values

# Function to Read, Filter, Save Chunks of Dataset

The fix is in the body of `importStateChunk`: replace the in-loop `pd.concat` with list-append, then concat once. Data cleaning is also included in this block.

In [ ]:
def importStateChunk(state, fullNPPES_path, cols):
    # Establishing the results file path
    results_path = f"/Users/lukebincarousky/Downloads/NPPES/NPPES_Data_Dissemination_May_2026_V2/V2 States/{state} NPPES Extract.csv"

    # Collect matching chunks into a list, then concat once at the end (O(n) instead of O(n^2)).
    chunk_list = []
    chunksize = 100000
    for each_chunk in pd.read_csv(fullNPPES_path, chunksize=chunksize, usecols=cols, low_memory=False):
        filtered_chunk = each_chunk[each_chunk['Provider Business Practice Location Address State Name'] == state]
        if not filtered_chunk.empty:
            chunk_list.append(filtered_chunk)

    if chunk_list:
        filtered_df = pd.concat(chunk_list, ignore_index=True)
    else:
        # No rows for this state — produce an empty frame with the expected columns
        filtered_df = pd.DataFrame(columns=cols)

    # Dropping columns with all null values
    clean_df = filtered_df.dropna(axis=1, how='all')
    clean_df = clean_df.reset_index(drop=True)

    # Cleaning specified columns
    # ZipCodes to 5 digits
    clean_df['Provider Business Mailing Address Postal Code'] = clean_df['Provider Business Mailing Address Postal Code'].astype(str).str[:5]
    clean_df['Provider Business Practice Location Address Postal Code'] = clean_df['Provider Business Practice Location Address Postal Code'].astype(str).str[:5]

    # Phone numbers to 10 digits
    clean_df['Provider Business Mailing Address Telephone Number'] = clean_df['Provider Business Mailing Address Telephone Number'].astype(str).str[:10]
    clean_df['Provider Business Mailing Address Fax Number'] = clean_df['Provider Business Mailing Address Fax Number'].astype(str).str[:10]
    clean_df['Provider Business Practice Location Address Telephone Number'] = clean_df['Provider Business Practice Location Address Telephone Number'].astype(str).str[:10]
    clean_df['Provider Business Practice Location Address Fax Number'] = clean_df['Provider Business Practice Location Address Fax Number'].astype(str).str[:10]
    clean_df['Authorized Official Telephone Number'] = clean_df['Authorized Official Telephone Number'].astype(str).str[:10]

    # Mapping County from MSA Workbook based on ZipCode
    clean_df = clean_df.merge(
        msa_geo[['Zip', 'County Name']].assign(Zip=msa_geo['Zip'].astype(str).str.zfill(5)),
        left_on='Provider Business Practice Location Address Postal Code',
        right_on='Zip',
        how='left'
    )

    # Rename Zip as ZipCode and County Name as County
    clean_df.rename(columns={'Zip': 'ZipCode', 'County Name': 'County'}, inplace=True)

    # Mapping Specialties via Taxonomy Codes
    # Taxonomy Code_1
    clean_df = clean_df.merge(
        spec_dict[['Code', 'Grouping', 'Classification', 'Display Name']].assign(Code=spec_dict['Code']),
        left_on='Healthcare Provider Taxonomy Code_1',
        right_on='Code',
        how='left'
    )

    # Rename Columns with _1 suffix to avoid confusion with future merges
    clean_df.rename(columns={'Code': 'Code_1', 'Grouping': 'Grouping_1', 'Classification': 'Class_1', 'Display Name': 'Spec_1'}, inplace=True)
    
    # Taxonomy Code_2
    clean_df = clean_df.merge(
        spec_dict[['Code', 'Grouping', 'Classification', 'Display Name']].assign(Code=spec_dict['Code']),
        left_on='Healthcare Provider Taxonomy Code_1',
        right_on='Code',
        how='left'
    )

    # Rename Columns with _2 suffix to avoid confusion with future merges
    clean_df.rename(columns={'Code': 'Code_2', 'Grouping': 'Grouping_2', 'Classification': 'Class_2', 'Display Name': 'Spec_2'}, inplace=True)

    # Taxonomy Code_3
    clean_df = clean_df.merge(
        spec_dict[['Code', 'Grouping', 'Classification', 'Display Name']].assign(Code=spec_dict['Code']),
        left_on='Healthcare Provider Taxonomy Code_1',
        right_on='Code',
        how='left'
    )

    # Rename Columns with _3 suffix to avoid confusion with future merges
    clean_df.rename(columns={'Code': 'Code_3', 'Grouping': 'Grouping_3', 'Classification': 'Class_3', 'Display Name': 'Spec_3'}, inplace=True)

    # Taxonomy Code_4
    clean_df = clean_df.merge(
        spec_dict[['Code', 'Grouping', 'Classification', 'Display Name']].assign(Code=spec_dict['Code']),
        left_on='Healthcare Provider Taxonomy Code_1',
        right_on='Code',
        how='left'
    )

    # Rename Columns with _4 suffix to avoid confusion with future merges
    clean_df.rename(columns={'Code': 'Code_4', 'Grouping': 'Grouping_4', 'Classification': 'Class_4', 'Display Name': 'Spec_4'}, inplace=True)

    # Taxonomy Code_5
    clean_df = clean_df.merge(
        spec_dict[['Code', 'Grouping', 'Classification', 'Display Name']].assign(Code=spec_dict['Code']),
        left_on='Healthcare Provider Taxonomy Code_1',
        right_on='Code',
        how='left'
    )

    # Rename Columns with _5 suffix to avoid confusion with future merges
    clean_df.rename(columns={'Code': 'Code_5', 'Grouping': 'Grouping_5', 'Classification': 'Class_5', 'Display Name': 'Spec_5'}, inplace=True)

    # Saving the cleaned DataFrame to a new CSV file
    clean_df.to_csv(results_path, index=False)

    msg = f"Files Saved for {state}"
    return msg

# Looping Through States

In [4]:
# List of items to remove
items_to_remove = ['PR', 'VI', np.nan, 'AS', 'GU', 'PW', 'FM', 'MP', 'MH']

# Removing nan & territories
state_list = [state for state in state_list if state not in items_to_remove]

In [5]:
# The list below loops through each state
for each_state in state_list:
    # The line below invokes our function
    result = importStateChunk(each_state, fullNPPES_path, cols)
    print(result)

UnboundLocalError: cannot access local variable 'df' where it is not associated with a value